In [9]:
from datasets import load_from_disk
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import classification_report
import torch
from torch.nn.utils.rnn import pad_sequence

# -------------------------------------------------------
# Load model & tokenizer
# -------------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained("../model/final")
tokenizer = AutoTokenizer.from_pretrained("../model/final")

# -------------------------------------------------------
# Load dataset
# -------------------------------------------------------
eval_dataset = load_from_disk("../data/tokenized/eval/")

if "text" in eval_dataset.column_names:
    eval_dataset = eval_dataset.remove_columns(["text"])

# -------------------------------------------------------
# Helper kolasi batch manual
# -------------------------------------------------------
def collate_fn(batch):
    input_ids = [torch.tensor(x["input_ids"]) for x in batch]
    attention_mask = [torch.tensor(x["attention_mask"]) for x in batch]
    labels = [torch.tensor(x['label']) for x in batch]

    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = torch.stack(labels)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

# -------------------------------------------------------
# DataLoader
# -------------------------------------------------------
loader = torch.utils.data.DataLoader(
    eval_dataset,
    batch_size=32,
    collate_fn=collate_fn
)

# -------------------------------------------------------
# Evaluasi
# -------------------------------------------------------
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in loader:
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        preds = outputs.logits.argmax(-1)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(batch["labels"].cpu().tolist())

print(classification_report(all_labels, all_preds, target_names=["normal", "judol"]))


              precision    recall  f1-score   support

      normal       0.98      0.98      0.98       603
       judol       0.98      0.98      0.98       654

    accuracy                           0.98      1257
   macro avg       0.98      0.98      0.98      1257
weighted avg       0.98      0.98      0.98      1257

